# DATA401 Final Map Project - OSM Data Pull & Preparation
**GIS-Based Flood Risk Assessment of Student Housing in Manila's University Belt**

Guillarte, Dana Louise | So, Chrysille Grace | April 2026

---

This notebook handles all OpenStreetMap data collection and pre-processing for the flood risk assessment. It runs through eight sequential steps:

1. Environment setup and bounding box definition
2. Universities pull (used as anchors for the proximity filter)
3. Raw student housing pull (dormitories, apartments, residential buildings)
4. 800m proximity filter to isolate likely student housing
5. Road network pull via OSMnx
6. Waterway pull and coverage check
7. Building levels, waterway vulnerability, and density grid computation
8. Flood hazard overlay and composite vulnerability assembly

## Step 1 - Environment Setup

Imports all required libraries and creates the output folder structure. The bounding box is defined here as a Shapely polygon using the study area coordinates:

| Edge  | Coordinate |
|-------|------------|
| South | 14.5750    |
| North | 14.6100    |
| West  | 120.9850   |
| East  | 121.0150   |

This covers Malate, Ermita, Paco, Pandacan, and Sampaloc. This captures DLSU, PUP, UST, PLM, Mapua, FEU, CEU, and San Sebastian while excluding Makati.

**Note:** `box()` from Shapely takes arguments as `(west, south, east, north)`. OSMnx's `features_from_polygon` accepts the polygon directly, so the coordinate order is handled internally — no manual reordering needed when using this method.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box as shapely_box

# Create folders
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Bounding box
bbox_polygon = shapely_box(120.9850, 14.5750, 121.0150, 14.6100)

print("All imports loaded.")

All imports loaded.


## Step 2 - Pull University & College Locations (Component 3)

University locations serve two roles in this project:
1. **Proximity filter anchor** — buildings within 800m of a university are retained as student housing in Step 4
2. **Routing destinations** — used in the road accessibility analysis to check whether housing areas can still reach campus during a flood

Both `amenity=university` and `amenity=college` are queried to capture all relevant institutions. This matters because some entries in OSM (e.g., individual UST colleges) are tagged as `college` rather than `university`. Large campuses like UST are also mapped as **polygon relations**, so the query must return all geometry types — nodes, ways, and relations.

Result is saved to `data/raw/universities.geojson` for use in subsequent steps.

In [ ]:
# Pull university and college locations from OSM
universities = ox.features_from_polygon(
    bbox_polygon,
    tags={"amenity": ["university", "college"]}
)

print(universities[["name", "amenity", "geometry"]].head(20))
print(f"Total universities found: {len(universities)}")

universities.to_file("data/raw/universities.geojson", driver="GeoJSON")
print("Universities saved.")

                                                                   name  \
element  id                                                               
node     255050283       Manila Central University College of Dentistry   
         255068563                            Manila Central University   
         255071563                           De Ocampo Memorial College   
         1732605874                          UST Faculty of Engineering   
         1732605876                 UST College of Fine Arts and Design   
         1974721543               Sudeco Polytechnic Integrated College   
         1974746063                             Access Computer College   
         1974746081                     Montessori Professional College   
         2545181890                         UST College of Architecture   
         4331538009               University of the East College of Law   
         4723440689                         FEU Institute of Technology   
         5205122121      

## Step 3 - Pull Raw Housing Buildings (Component 1)

Student housing in Manila is rarely tagged precisely in OSM. Most dorms and boarding houses appear under generic tags like `building=apartments` or `building=residential` rather than `building=dormitory`. This pull casts a wide net using four tags:

- `building=dormitory` — explicitly tagged dorms
- `building=apartments` — multi-unit residential buildings
- `building=residential` — general residential buildings
- `amenity=student_accommodation` — explicitly tagged student housing

In [ ]:
# Pull potential student housing buildings from OSM
housing = ox.features_from_polygon(
    bbox_polygon,
    tags={
        "building": ["dormitory", "apartments", "residential"],
        "amenity": "student_accommodation"
    }
)

print(f"Total buildings before filter: {len(housing)}")
print(housing.geometry.geom_type.value_counts())

housing.to_file("data/raw/housing_raw.geojson", driver="GeoJSON")
print("Raw housing saved.")

Total buildings before filter: 313
Polygon    310
Point        3
Name: count, dtype: int64
Raw housing saved.


## Step 4 - Apply 800m Proximity Filter

Because building tags alone cannot reliably identify student housing, a spatial filter is applied: only buildings within **800 metres of a university or college** are kept. 800m is a reasonable walking distance for students and is more robust than relying on name matching or tags.

1. Both layers are reprojected to **EPSG:32651** so distances are in metres, not degrees
2. All university geometries are merged into a single union and buffered by 800m
3. Any housing building whose footprint intersects this buffer is retained

In [ ]:
# Reproject both layers to EPSG:32651 for metre-based distance calculations
housing_proj = housing.to_crs(epsg=32651)
universities_proj = universities.to_crs(epsg=32651)

# Merge all university geometries into one shape, then buffer by 800m
uni_buffer = universities_proj.geometry.union_all().buffer(800)

# Keep only housing buildings that fall within the 800m buffer
student_housing = housing_proj[housing_proj.geometry.intersects(uni_buffer)].copy()

print(f"Before filter: {len(housing_proj)}")
print(f"After 800m filter: {len(student_housing)}")  # should be 294

Before filter: 313
After 800m filter: 294


## Step 5 - Pull Road Network

The road network is pulled via OSMnx as a **directed graph** (`network_type="drive"`), which supports shortest-path routing with NetworkX. It serves two roles:

- **Exposure layer** - road segments are intersected with flood hazard zones to flag which roads get inundated
- **Accessibility analysis** - flooded segments are removed from the graph and routing is re-run to identify which housing areas lose access to universities

The graph is converted to GeoDataFrames (`nodes`, `edges`) for spatial operations. Edge geometries represent individual road segments, which are the units used in the flood overlay later.

The output is saved to `data/raw/roads.geojson`.

In [ ]:
# Pull drivable road network as a graph for routing analysis
G = ox.graph_from_polygon(bbox_polygon, network_type="drive")

# Convert graph to GeoDataFrames for spatial operations
nodes, edges = ox.graph_to_gdfs(G)

print(f"Nodes: {len(nodes)}")
print(f"Road segments: {len(edges)}")

edges.to_file("data/raw/roads.geojson", driver="GeoJSON")
print("Roads saved.")

Nodes: 1487
Road segments: 3599
Roads saved.


## Step 6 - Pull Waterways and Assess Coverage

Proximity to waterways (esteros, canals, rivers) is included as a supplementary vulnerability indicator, buildings closer to drainage channels tend to be at higher flood risk. This component was flagged as uncertain because drainage infrastructure is often under-mapped in OSM for Manila.

The query targets four waterway types: `river`, `stream`, `canal`, and `drain`.

**Decision criteria:**
- **Include** if major esteros (Estero de Paco, Estero de Tripa de Gallina) and the Pasig River appear clearly
- **Drop** if results are sparse or key canals are missing

In [ ]:
# Pull waterway features from OSM (rivers, streams, canals, drains)
waterways = ox.features_from_polygon(
    bbox_polygon,
    tags={"waterway": ["river", "stream", "canal", "drain"]}
)

print(f"Waterway features found: {len(waterways)}")
print(waterways[["name", "waterway"]].dropna(subset=["name"]).head(20))

waterways.to_file("data/raw/waterways.geojson", driver="GeoJSON")
print("Waterways saved.")

Waterway features found: 96
                                        name waterway
element id                                           
way     4304724               Estero de Paco    river
        4443915                  Pasig River    river
        4940654   Estero de Tripa de Gallina    river
        5055435           Estero de Pandacan    river
        5055440           Estero de Pandacan    river
        12805235            Estero de Aviles   stream
        22730074          Estero de Sampaloc    river
        22742600  Estero de Tripa de Gallina    river
        22742601  Estero de Tripa de Gallina    river
        27275946            Estero de Quiapo    river
        27276016           Estero de Uli-Uli   stream
        27276237            Estero de Quiapo    canal
        27276489               Perlita Creek   stream
        27296235                 Pasig River    river
        27323169            Estero de Balete    river
        27323219         Estero de Concordia    river


## Step 7 - All Buildings Pull & Building Levels Assessment

This step does two things:

**7a - All-buildings pull (for density grid)**
All buildings in the study area are pulled, regardless of type. This full set is used to compute the **grid-based building density** indicator (Step 10), which counts how many student housing buildings fall in each 200m × 200m cell.

**7b - Building levels coverage check**
`building:levels` (floor count) is the primary vulnerability indicator. Lower buildings are treated as more flood-vulnerable since floodwater affects a greater proportion of the structure. Coverage across the full building stock is only 9.4%, but coverage within the student housing subset is **68%** (200 of 294 buildings), which is sufficient to use this indicator. The remaining 32% will be imputed with the dataset median.

In [ ]:
# Pull all buildings in the study area (used for grid-based density analysis)
all_buildings = ox.features_from_polygon(
    bbox_polygon,
    tags={"building": True}
)

print(f"Total buildings in study area: {len(all_buildings)}")

# Check how many buildings have the levels tag across the full dataset
has_levels = all_buildings["building:levels"].notna().sum()
total = len(all_buildings)
print(f"Buildings with levels tag: {has_levels}/{total} ({100*has_levels/total:.1f}%)")

# Check levels coverage specifically within the student housing subset
if "building:levels" in student_housing.columns:
    sh_has_levels = student_housing["building:levels"].notna().sum()
    print(f"Student housing with levels: {sh_has_levels}/{len(student_housing)} ({100*sh_has_levels/len(student_housing):.1f}%)")

all_buildings.to_file("data/raw/all_buildings.geojson", driver="GeoJSON")
print("All buildings saved.")

Total buildings in study area: 28892
Buildings with levels tag: 2708/28892 (9.4%)
Student housing with levels: 200/294 (68.0%)
All buildings saved.


## Step 8 - Building Levels: Imputation & Vulnerability Score

For the 94 student housing buildings missing `building:levels`, the **median floor count (5 floors)** from the rest of the dataset is filled in. These imputed rows are flagged with `levels_imputed=True` so the imputation is transparent in the final output.

The vulnerability score from floor count is computed as:

**vuln_levels = 1 - (levels / max_levels)**

This gives lower buildings a **higher** vulnerability score, which reflects the assumption that a 2-storey building is more severely impacted by flooding than a 20-storey one where only the ground floor is at risk.

In [ ]:
# Convert building:levels to numeric, coercing any non-numeric values to NaN
student_housing["levels"] = pd.to_numeric(
    student_housing["building:levels"], errors="coerce"
)

# Check distribution before filling missing values
print("Levels distribution (before imputation):")
print(student_housing["levels"].describe())
print(f"\nMissing: {student_housing['levels'].isna().sum()}/{len(student_housing)}")

# Fill missing levels with the dataset median and flag imputed rows
median_levels = student_housing["levels"].median()
student_housing["levels_imputed"] = student_housing["levels"].isna()
student_housing["levels"] = student_housing["levels"].fillna(median_levels)

# Compute vulnerability score, lower buildings score higher (more flood-exposed)
max_lev = student_housing["levels"].max()
student_housing["vuln_levels"] = 1 - (student_housing["levels"] / max_lev)

print(f"\nMedian levels used for imputation: {median_levels}")
print(f"Imputed rows: {student_housing['levels_imputed'].sum()}")
print(f"Count after levels processing: {len(student_housing)}")  # must be 294

Levels distribution (before imputation):
count    200.00000
mean       7.40000
std        7.10934
min        1.00000
25%        3.00000
50%        5.00000
75%       10.00000
max       52.00000
Name: levels, dtype: float64

Missing: 94/294

Median levels used for imputation: 5.0
Imputed rows: 94
Count after levels processing: 294


## Step 9 - Waterway Distance & Vulnerability Score

Distance from each student housing building to the nearest waterway is computed using the unioned waterway geometry. Buildings immediately adjacent to an estero or canal receive a higher vulnerability score; buildings further away receive a lower one.

The score is normalized to a 0–1 range:

**vuln_waterway = 1 - (dist_waterway_m / max_dist)**

In [ ]:
# Reproject waterways to EPSG:32651 for metre-based distance calculations
waterways_proj = waterways.to_crs(epsg=32651)

# Merge all waterway geometries into one shape for efficient distance computation
waterways_union = waterways_proj.geometry.union_all()

# Compute distance from each housing building to the nearest waterway
student_housing["dist_waterway_m"] = student_housing.geometry.apply(
    lambda geom: geom.distance(waterways_union)
)

# Normalize: closer to waterway = higher vulnerability score
max_dist = student_housing["dist_waterway_m"].max()
student_housing["vuln_waterway"] = 1 - (student_housing["dist_waterway_m"] / max_dist)

print("Waterway distances computed.")
print(student_housing["dist_waterway_m"].describe())
print(f"\nCount after waterway processing: {len(student_housing)}")  # must be 294

Waterway distances computed.
count    294.000000
mean     146.539888
std      182.266312
min        0.000000
25%       36.263751
50%       84.957399
75%      176.619023
max      957.397091
Name: dist_waterway_m, dtype: float64

Count after waterway processing: 294


## Step 10 - Grid-Based Building Density (Exposure Indicator)

A **200m × 200m fishnet grid** is placed over the student housing extent. The number of student housing buildings in each cell is counted via a spatial join. This becomes the **density exposure indicator** in the composite risk index.

In [ ]:
# Get the spatial extent of the student housing layer
xmin, ymin, xmax, ymax = student_housing.total_bounds
cell_size = 200  # 200m x 200m grid cells

# Generate grid cell corner coordinates
cols = np.arange(xmin, xmax, cell_size)
rows = np.arange(ymin, ymax, cell_size)

# Build a list of square polygon cells covering the full extent
grid_cells = [
    shapely_box(x, y, x + cell_size, y + cell_size)
    for x in cols for y in rows
]

# Create GeoDataFrame from grid cells and assign unique cell IDs
grid = gpd.GeoDataFrame({'geometry': grid_cells}, crs=student_housing.crs)
grid["cell_id"] = range(len(grid))

# Count how many student housing buildings fall within each cell
joined = gpd.sjoin(student_housing, grid[["geometry", "cell_id"]], how='left', predicate='intersects')
counts = joined.groupby('cell_id').size().reset_index(name='building_count')
grid = grid.merge(counts, on='cell_id', how='left')
grid['building_count'] = grid['building_count'].fillna(0)  # cells with no buildings → 0

print(f"Grid cells: {len(grid)}")
print(f"Max buildings in a single cell: {grid['building_count'].max()}")
print(f"Cells with at least 1 building: {(grid['building_count'] > 0).sum()}")

grid.to_file("data/processed/density_grid.geojson", driver="GeoJSON")
print("Density grid saved.")

Grid cells: 340
Max buildings in a single cell: 34.0
Cells with at least 1 building: 96
Density grid saved.


## Step 11 - Join Grid Density Back to Buildings

Each student housing building is assigned the `building_count` of the grid cell it falls in. This makes the density score available as a per-building attribute for the composite risk index.

A deduplication step handles edge cases where a building footprint intersects two adjacent grid cells — the first match is kept in those cases.

Running count check: should remain **294 buildings** throughout.

In [11]:
# Reset index first to avoid MultiIndex issues
student_housing = student_housing.reset_index(drop=True)

# Find which cell each building falls in
joined_density = gpd.sjoin(
    student_housing[["geometry"]],
    grid[["geometry", "cell_id", "building_count"]],
    how="left",
    predicate="intersects"
)

# Deduplicate in case any building hit multiple cells
joined_density = joined_density[~joined_density.index.duplicated(keep="first")]

# Add building_count to student_housing
student_housing["building_count"] = joined_density["building_count"].values
student_housing["building_count"] = student_housing["building_count"].fillna(0)

print(f"Count after density join: {len(student_housing)}")  # must be 294
print(f"Building count sample:\n{student_housing['building_count'].describe()}")

Count after density join: 294
Building count sample:
count    294.000000
mean      12.278912
std       11.529918
min        1.000000
25%        4.000000
50%        7.000000
75%       21.000000
max       34.000000
Name: building_count, dtype: float64


## Step 12 - Load Flood Hazard Data

The flood hazard layer comes from **Project NOAH's 100-year return period flood map** for Metro Manila. This represents a worst-case scenario and is the primary hazard input for the entire analysis.

The raw shapefile uses a numeric `Var` field for hazard classification, which is mapped to descriptive labels:

| `Var` value | Hazard level |
|-------------|--------------|
| 1.0         | Low          |
| 2.0         | Medium       |
| 3.0         | High         |

The file is loaded from `Data/Metro Manila/MetroManila_Flood_100year.shp`.

In [ ]:
# Load Project NOAH 100-year return period flood hazard shapefile
flood_raw = gpd.read_file("Data/Metro Manila/MetroManila_Flood_100year.shp")

# Map numeric Var field to descriptive hazard labels and keep raw score for sorting
flood_raw["hazard_level"] = flood_raw["Var"].map({
    1.0: "low",
    2.0: "medium",
    3.0: "high"
})
flood_raw["hazard_score"] = flood_raw["Var"]

print("Hazard label distribution:")
print(flood_raw[["Var", "hazard_level"]].to_string())

Hazard label distribution:
   Var hazard_level
0  1.0          low
1  2.0       medium
2  3.0         high


## Step 13 - Clip Flood Hazard to Study Area

The Project NOAH flood hazard shapefile covers all of Metro Manila. It is clipped to the study area bounding box to reduce processing overhead and keep results focused on the university belt.

Medium hazard is the dominant classification across the study area, which is consistent with the relatively flat, low-lying terrain of the Manila university belt. The clipped layer is saved to `data/processed/flood_hazard.geojson`.

In [ ]:
# Reproject flood hazard layer to EPSG:32651 to match all other layers
flood_proj = flood_raw.to_crs(epsg=32651)

# Define the study area bounding box and reproject to EPSG:32651 for clipping
study_bbox = shapely_box(120.9850, 14.5750, 121.0150, 14.6100)
study_bbox_proj = gpd.GeoDataFrame(geometry=[study_bbox], crs="EPSG:4326").to_crs(epsg=32651)

# Clip flood hazard to study area extent
flood_clipped = gpd.clip(flood_proj, study_bbox_proj)

# Check how much area each hazard level covers within the study area
print(f"Features after clipping to study area: {len(flood_clipped)}")
flood_clipped["area_km2"] = flood_clipped.geometry.area / 1_000_000
print("\nArea coverage by hazard level:")
print(flood_clipped.groupby("hazard_level")["area_km2"].sum().round(3))

flood_clipped.to_file("data/processed/flood_hazard.geojson", driver="GeoJSON")
print("\nFlood hazard saved.")

Features after clipping to study area: 3

Area coverage by hazard level:
hazard_level
high      1.020
low       2.713
medium    5.846
Name: area_km2, dtype: float64

Flood hazard saved.


## Step 14 - Intersect Student Housing with Flood Hazard Zones

Each student housing building is spatially joined to the flood hazard layer to determine which hazard zone it falls in. Where a building overlaps multiple zones (e.g., a large footprint straddling a low and medium zone), the **highest hazard score is kept**. This is handled by sorting descending on `hazard_score` before deduplication.

**274 of 294 buildings (93%)** fall within a mapped flood hazard zone. The 20 buildings with no hazard classification are assigned `hazard_score=0` and `in_flood_zone=False`.

In [ ]:
# Join each student housing building to the flood hazard zone it intersects
housing_flood = gpd.sjoin(
    student_housing,
    flood_clipped[["hazard_level", "hazard_score", "geometry"]],
    how="left",
    predicate="intersects"
)

print(f"After flood join (before dedup): {len(housing_flood)}")

# Sort by hazard_score descending so the highest hazard is kept after deduplication
housing_flood["geom_hash"] = housing_flood.geometry.apply(lambda g: hash(g.wkt))
housing_flood = housing_flood.sort_values("hazard_score", ascending=False)
housing_flood = housing_flood.drop_duplicates(subset=["geom_hash"], keep="first")
housing_flood = housing_flood.drop(columns=["geom_hash", "index_right", "cell_id"], errors="ignore")

# Buildings outside all hazard zones get score 0 and are flagged accordingly
housing_flood["hazard_level"] = housing_flood["hazard_level"].fillna("none")
housing_flood["hazard_score"] = housing_flood["hazard_score"].fillna(0)
housing_flood["in_flood_zone"] = housing_flood["hazard_score"] > 0

print(f"After dedup: {len(housing_flood)}")  # must be 294
print("\nHazard distribution:")
print(housing_flood["hazard_level"].value_counts())
print(f"\nBuildings in any flood zone: {housing_flood['in_flood_zone'].sum()}/{len(housing_flood)}")

After flood join (before dedup): 454
After dedup: 294

Hazard distribution:
hazard_level
medium    221
low        30
high       23
none       20
Name: count, dtype: int64

Buildings in any flood zone: 274/294


## Step 15 - Check Building Type Distribution

A quick check on the `building` tag values across the 294 student housing buildings. This informs the vulnerability score assigned in the next step.

As expected, explicitly tagged dormitories are a minority. The majority are generic residential or apartment tags — this is why the proximity filter in Step 4 was necessary to isolate likely student housing rather than relying on tags alone.

In [15]:
# Check building types present in dataset
print("Building types in student housing:")
print(housing_flood["building"].value_counts())

Building types in student housing:
building
residential    178
apartments      88
dormitory       28
Name: count, dtype: int64


## Step 16 - Building Type Vulnerability Score

A vulnerability score is assigned based on building type. Dormitories are weighted highest because they house the most students per building and represent the greatest concentration of at-risk occupants. The scores reflect relative occupancy risk, not structural fragility.

Any building types not in the mapping table are filled with the median score. All 294 buildings are covered with no missing values.

In [ ]:
# Vulnerability scores by building type
type_score_map = {
    "dormitory": 1.0,
    "student_accommodation": 0.9,
    "apartments": 0.6,
    "residential": 0.4
}

# Map scores onto each building based on its type tag
housing_flood["vuln_type"] = housing_flood["building"].map(type_score_map)

# Fill any building types not in the map with the median score
median_type_score = housing_flood["vuln_type"].median()
housing_flood["vuln_type"] = housing_flood["vuln_type"].fillna(median_type_score)

print("Building type vulnerability scores:")
print(housing_flood.groupby("building")["vuln_type"].first().sort_values(ascending=False))
print(f"\nVuln_type summary:")
print(housing_flood["vuln_type"].describe())
print(f"\nCount: {len(housing_flood)}")  # must be 294

Building type vulnerability scores:
building
dormitory      1.0
apartments     0.6
residential    0.4
Name: vuln_type, dtype: float64

Vuln_type summary:
count    294.000000
mean       0.517007
std        0.180774
min        0.400000
25%        0.400000
50%        0.400000
75%        0.600000
max        1.000000
Name: vuln_type, dtype: float64

Count: 294


## Step 17 - Building Footprint Area

Footprint area (m²) is computed directly from the polygon geometry. It is always available regardless of how well-tagged a building is in OSM. It serves as a supplementary attribute that gives a rough sense of building scale and is included in the final output for reference.

The normalized value (`footprint_norm`) scales area to a 0–1 range relative to the largest building in the dataset (3,660 m²). Footprint area is **not** used as a weighted input in the composite risk score. It is retained as a descriptive attribute.

In [ ]:
# Footprint area
housing_flood["footprint_area_m2"] = housing_flood.geometry.area

print("Footprint area summary (m²):")
print(housing_flood["footprint_area_m2"].describe())

# Normalize to 0-1 for use as supplementary attribute
max_area = housing_flood["footprint_area_m2"].max()
housing_flood["footprint_norm"] = housing_flood["footprint_area_m2"] / max_area

print(f"\nLargest building footprint: {max_area:.1f} m²")
print(f"Count: {len(housing_flood)}")  # must be 294

Footprint area summary (m²):
count     294.000000
mean      502.613904
std       542.080203
min         0.000000
25%       148.024179
50%       328.402946
75%       663.717007
max      3660.781620
Name: footprint_area_m2, dtype: float64

Largest building footprint: 3660.8 m²
Count: 294


## Step 18 - Transfer Waterway Scores to Final Housing Dataset

The waterway vulnerability scores computed in Step 9 were attached to `student_housing`. At this point, the active dataset is `housing_flood` (which includes the flood hazard join from Step 14). This cell transfers `vuln_waterway` and `dist_waterway_m` across by direct index alignment. Both datasets contain the same 294 buildings in the same order.

In [18]:
# Pull waterway vulnerability scores into housing_flood 
housing_flood = housing_flood.reset_index(drop=True)
student_housing_reset = student_housing.reset_index(drop=True)

# Since both came from the same 294 buildings in the same order, assign directly
housing_flood["vuln_waterway"] = student_housing_reset["vuln_waterway"].values
housing_flood["dist_waterway_m"] = student_housing_reset["dist_waterway_m"].values

print(f"Waterway vulnerability transferred: {housing_flood['vuln_waterway'].notna().sum()}/{len(housing_flood)}")
print(f"Count: {len(housing_flood)}")  # must be 294

Waterway vulnerability transferred: 294/294
Count: 294


## Step 19 - Vulnerability Column Completeness Check

A final verification that all required vulnerability columns are present and fully populated before saving. Any missing values here would silently produce incorrect risk scores in the composite index step.

Expected columns with zero missing values:
- `vuln_levels` - floor count vulnerability (median-imputed where missing)
- `vuln_type` - building type vulnerability
- `vuln_waterway` - distance-to-waterway vulnerability
- `footprint_area_m2` - raw footprint area
- `footprint_norm` - normalized footprint area

In [ ]:
# List of all vulnerability columns that must be present before saving
vuln_cols = ["vuln_levels", "vuln_type", "vuln_waterway", "footprint_area_m2", "footprint_norm"]

# Check each column exists and has no missing values
print("Vulnerability columns check:")
for col in vuln_cols:
    if col in housing_flood.columns:
        missing = housing_flood[col].isna().sum()
        print(f"  {col}: OK — {missing} missing values")
    else:
        print(f"  {col}: MISSING — not computed yet")

print(f"\nFinal count: {len(housing_flood)}")  # must be 294

Vulnerability columns check:
  vuln_levels: OK — 0 missing values
  vuln_type: OK — 0 missing values
  vuln_waterway: OK — 0 missing values
  footprint_area_m2: OK — 0 missing values
  footprint_norm: OK — 0 missing values

Final count: 294


## Step 20 - Save Final Dataset

All vulnerability indicators, flood hazard attributes, and building metadata are saved together to `data/processed/student_housing_final.geojson`. This is the input file for the composite risk index and all subsequent mapping steps.

Below is a summary of what each building record now contains:
- Geometry (polygon footprint in EPSG:32651)
- OSM attributes (`building`, `building:levels`, `name`, etc.)
- `levels` and `levels_imputed` - floor count with imputation flag
- `vuln_levels` - floor-count vulnerability score (0–1)
- `dist_waterway_m` and `vuln_waterway` - waterway proximity
- `building_count` - housing density in the surrounding 200m cell
- `hazard_level` and `hazard_score` - flood hazard classification
- `in_flood_zone` - boolean flag
- `vuln_type` - building type vulnerability score
- `footprint_area_m2` and `footprint_norm` - building footprint size

In [ ]:
housing_flood.to_file("data/processed/student_housing_final.geojson", driver="GeoJSON")
print("student_housing_final.geojson saved with all vulnerability indicators.")
print(f"Columns: {list(housing_flood.columns)}")

student_housing_final.geojson saved with all vulnerability indicators.
Columns: ['geometry', 'building', 'building:levels', 'name', 'tourism', 'amenity', 'addr:city', 'addr:province', 'addr:street', 'addr:street:corner', 'source', 'height', 'addr:postcode', 'roof:shape', 'addr:housenumber', 'building:form', 'building:material', 'feature:category', 'ground:slope', 'landscape', 'plinthlevel:height', 'roof:material', 'visual:condition', 'building:colour', 'basement', 'note', 'building:architecture', 'heritage', 'description', 'addr2:street', 'internet_access', 'operator', 'capacity', 'cuisine', 'delivery', 'facebook', 'opening_hours', 'payment:cash', 'phone', 'smoking', 'takeaway', 'alt_name', 'roof:levels', 'building:type', 'building:levels:underground', 'roof:colour', 'mapillary', 'contact:phone', 'check_date', 'wheelchair', 'building:min_level', 'email', 'website', 'access', 'lit', 'maxheight:physical', 'park_ride', 'parking', 'supervised', 'layer', 'wikidata', 'type', 'old_name', 'wik